# NFL pick'em odds

Scrapes the Las Vegas odds table from vegasinsider.com and averages the point spread across books.
Negative spread = favored (the more negative, the bigger the expected win).

The site's HTML is flaky — columns get scrambled, junk like `--4.5` shows up, and the spread /
total / moneyline sections are all stacked into one table — so the parsing below validates every
cell and silently drops whatever doesn't make sense instead of crashing.

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', None)

In [2]:
url = "https://www.vegasinsider.com/nfl/odds/las-vegas/"
tab = pd.read_html(url)[0]

In [3]:
# All parsing / cleaning functions live in pickem.py (shared with email_picks.py, which
# emails Hannah's Entry A picks on a schedule). Edit them there.
from pickem import *


In [4]:
raw, sources = load_sections(tab)
full = raw.copy()
for s in sources:
    full[s] = raw[s].map(parse_line)

# classify sections by typical magnitude: spreads are small, moneylines are +-100 and up,
# and totals ('o47.5 ...') never parse with parse_line at all
med_abs = full.groupby('section')[sources].apply(lambda d: d.abs().median().median())
spread_sections = med_abs[med_abs < 50].index
ml_sections = med_abs[med_abs >= 100].index
total_sections = med_abs[med_abs.isna()].index

spreads = clean_spreads(full[full['section'].isin(spread_sections)], sources)
spreads['ave_spread'] = spreads[sources].mean(axis=1)
spreads['n_books'] = spreads[sources].notna().sum(axis=1)
spreads[['Team'] + sources + ['ave_spread', 'n_books']].sort_values('ave_spread')

HardRock: ignoring swapped spreads for ['Buccaneers', 'Bengals']
books split: Bills favored by ['Bet365', 'BetMGM', 'DraftKings', 'Caesars', 'FanDuel', 'HardRock', 'RiversCasino', 'Consensus'] | Texans favored by ['Fanatics']
books split: Packers favored by ['BetMGM', 'HardRock'] | Vikings favored by ['Bet365', 'DraftKings', 'Caesars', 'FanDuel', 'Fanatics', 'RiversCasino', 'Consensus']


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_spread,n_books
21,Chargers,-10.5,-10.5,-10.5,-10.5,-9.5,-10.5,-10.0,-10.0,-10.5,-10.277778,9
5,Jaguars,-7.5,-7.5,-7.5,-7.5,-7.5,-8.5,-8.5,-8.0,-7.5,-7.777778,9
19,Lions,-7.0,-7.0,-7.0,-7.0,-6.5,-6.5,-7.0,-7.0,-7.0,-6.888889,9
25,Eagles,-5.5,-4.5,-5.5,-4.5,-5.5,NaN,-5.0,-5.5,-5.5,-5.187500,8
3,Rams,-3.5,-3.5,-3.5,-3.5,-3.5,-4.5,-3.5,-3.5,-3.5,-3.611111,9
7,Bengals,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-4.0,-3.5,-3.5,-3.562500,8
1,Seahawks,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-3.5,-3.5,-3.5,-3.500000,8
23,Raiders,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-3.5,-3.5,-3.5,-3.500000,8
8,Ravens,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.500000,9
11,Steelers,-3.5,-3.0,-3.5,-3.0,-3.0,NaN,-3.5,-3.0,-3.5,-3.250000,8


## Moneyline → implied win probability

The moneyline section of the same table is literally the odds a team wins the matchup.
Each book's line is converted to an implied probability first (American odds are
nonlinear and asymmetric around ±100, so averaging them directly is biased), then the
probabilities are averaged across books and the vig removed by normalizing each game's
two probabilities to sum to 1. Swapped-team columns are dropped; genuine book
disagreement on toss-ups is kept.

In [5]:
ml = full[full['section'].isin(ml_sections)].reset_index(drop=True).copy()
raw_ml = raw[raw['section'].isin(ml_sections)].reset_index(drop=True)
for s in sources:
    ml[s] = raw_ml[s].map(parse_ml).map(lambda m: implied_prob(m) if not np.isnan(m) else np.nan)

# a swapped column reads p where consensus is 1-p; only decidable when consensus is
# >= 5 points from even (so |med - (1-med)| >= 0.10)
ml = drop_swapped(ml, sources, mirror=lambda p: 1 - p, min_consensus=0.10, label='moneylines')
ml = ml.dropna(subset=sources, how='all').reset_index(drop=True)
ml['q'] = ml[sources].mean(axis=1)                # vig-included average implied prob
ml['n_books'] = ml[sources].notna().sum(axis=1)
for g in range(0, len(ml) - 1, 2):
    tot = ml.loc[g, 'q'] + ml.loc[g + 1, 'q']
    ml.loc[[g, g + 1], 'win_prob'] = ml.loc[[g, g + 1], 'q'] / tot
ml[['Team'] + sources + ['q', 'win_prob', 'n_books']].round(3).sort_values('win_prob', ascending=False)

HardRock: ignoring swapped moneylines for ['Rams', 'Jaguars', 'Falcons', 'Saints', 'Giants']


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,q,win_prob,n_books
21,Chargers,0.846,0.867,0.846,0.855,0.855,0.852,0.846,0.848,0.846,0.851,0.814,9
5,Jaguars,0.796,0.810,0.798,0.796,0.800,NaN,0.800,0.794,0.798,0.799,0.767,8
19,Lions,0.759,0.778,0.753,0.765,0.770,0.765,0.767,0.775,0.753,0.765,0.733,9
25,Eagles,0.697,0.688,0.686,0.692,0.706,0.688,0.701,0.697,0.686,0.693,0.665,9
7,Bengals,0.672,0.667,0.658,0.672,0.672,0.655,0.683,0.672,0.658,0.668,0.641,9
3,Rams,0.667,0.667,0.664,0.663,0.664,NaN,0.667,0.672,0.664,0.666,0.639,8
1,Seahawks,0.667,0.667,0.636,0.655,0.664,0.655,0.655,0.667,0.636,0.656,0.628,9
23,Raiders,0.655,0.655,0.664,0.663,0.638,0.649,0.655,0.661,0.664,0.656,0.628,9
8,Ravens,0.643,0.667,0.636,0.645,0.640,0.649,0.643,0.655,0.636,0.646,0.619,9
11,Steelers,0.643,0.615,0.649,0.630,0.638,0.643,0.643,0.639,0.649,0.639,0.612,9


## Picks by matchup

Same numbers, but one line per game in the order the site lists them, so it's quick to
walk down the pick'em sheet. Format: `away (spread, win prob) @ home (spread, win prob) -> pick`.

In [6]:
sp = spreads.set_index('Team')['ave_spread']
wp = ml.set_index('Team')['win_prob']

games = full[full['section'].isin(spread_sections)].reset_index(drop=True)
for g in range(0, len(games) - 1, 2):
    away, home = games.loc[g, 'Team'], games.loc[g + 1, 'Team']
    pa, ph = wp.get(away, np.nan), wp.get(home, np.nan)
    if np.isnan(pa) and np.isnan(ph):
        continue                                  # game already final / no data
    pick = home if (ph if not np.isnan(ph) else -1) >= (pa if not np.isnan(pa) else -1) else away
    print(f"{away:>12} ({sp.get(away, np.nan):+5.1f}, {pa:4.0%})  @  "
          f"{home:<12} ({sp.get(home, np.nan):+5.1f}, {ph:4.0%})   ->  {pick}")

    Patriots ( +3.5,  37%)  @  Seahawks     ( -3.5,  63%)   ->  Seahawks
       49ers ( +3.6,  36%)  @  Rams         ( -3.6,  64%)   ->  Rams
      Browns ( +7.8,  23%)  @  Jaguars      ( -7.8,  77%)   ->  Jaguars
  Buccaneers ( +3.6,  36%)  @  Bengals      ( -3.6,  64%)   ->  Bengals
      Ravens ( -3.5,  62%)  @  Colts        ( +3.5,  38%)   ->  Ravens
     Falcons ( +3.2,  39%)  @  Steelers     ( -3.2,  61%)   ->  Steelers
       Bills ( -1.2,  51%)  @  Texans       ( +1.2,  49%)   ->  Bills
       Bears ( -2.6,  58%)  @  Panthers     ( +2.6,  42%)   ->  Bears
        Jets ( +1.9,  45%)  @  Titans       ( -1.9,  55%)   ->  Titans
      Saints ( +6.9,  27%)  @  Lions        ( -6.9,  73%)   ->  Lions
   Cardinals (+10.3,  19%)  @  Chargers     (-10.3,  81%)   ->  Chargers
    Dolphins ( +3.5,  37%)  @  Raiders      ( -3.5,  63%)   ->  Raiders
  Commanders ( +5.2,  33%)  @  Eagles       ( -5.2,  67%)   ->  Eagles
     Packers ( +0.7,  48%)  @  Vikings      ( -0.7,  52%)   ->  Vikings
 

## Two-entry strategy

**Entry A (season entry)**: vegas favorite in every game.
**Entry B (chalk with a fuse)**: same as A, except in toss-ups (|spread| < `TOSSUP`) take the
side the CBS public is *least* on. Once B falls off season-podium pace, set
`B_MODE = 'hunter'`: B then also flips the single game with the highest
edge = P_vegas(dog wins) × (CBS % on the favorite).

`cbs_pick_percents` is the public pick percentage on the **listed team** from the CBS pool page
(the "% picking" bar). Fill it in by hand until the scrape exists; games left out are
treated as unknown (Entry B falls back to chalk for them).

In [7]:
TOSSUP = 1.5          # |ave_spread| below this is a toss-up
B_MODE = 'fuse'       # 'fuse' (chalk + toss-up rule) or 'hunter' (fuse + one aggressive flip)

# public pick % on the named team, from the CBS pool page. One side per game is enough;
# the other side is inferred as 100 - x.
cbs_pick_percents = {   # Week 2, 2026-09-17
    'Bills': 91,
    'Texans': 63,
    'Patriots': 75,
    'Buccaneers': 97,
    'Panthers': 76,
    'Ravens': 97,
    'Bears': 89,
    'Packers': 85,
    'Eagles': 99,
    'Broncos': 54,
    'Chargers': 81,
    'Cowboys': 81,
    'Seahawks': 89,
    '49ers': 98,
    'Chiefs': 97,
    'Rams': 80,
}


In [8]:
games = full[full['section'].isin(spread_sections)].reset_index(drop=True)

# validate the hand-entered CBS dict against this week's team tags
teams = set(games['Team'])
bad = [t for t in cbs_pick_percents if t not in teams]
if bad:
    print(f"WARNING: cbs_pick_percents keys not in this week's teams (typo?): {bad}")
    print(f"         valid tags: {sorted(teams)}")
for g in range(0, len(games) - 1, 2):
    both = [t for t in games.loc[[g, g + 1], 'Team'] if t in cbs_pick_percents]
    if len(both) == 2 and sum(cbs_pick_percents[t] for t in both) != 100:
        print(f"WARNING: both sides entered for {both} and they don't sum to 100")
missing = [f"{games.loc[g, 'Team']}@{games.loc[g + 1, 'Team']}" for g in range(0, len(games) - 1, 2)
           if not any(t in cbs_pick_percents for t in games.loc[[g, g + 1], 'Team'])]
if missing:
    print(f"note: no CBS % for {missing}")
rows = []
for g in range(0, len(games) - 1, 2):
    away, home = games.loc[g, 'Team'], games.loc[g + 1, 'Team']
    pa, ph = wp.get(away, np.nan), wp.get(home, np.nan)
    if np.isnan(pa) or np.isnan(ph):
        continue
    fav, dog = (home, away) if ph >= pa else (away, home)
    p_fav = max(pa, ph)
    # CBS % on the favorite, from whichever side was entered
    if fav in cbs_pick_percents:
        cbs_fav = cbs_pick_percents[fav]
    elif dog in cbs_pick_percents:
        cbs_fav = 100 - cbs_pick_percents[dog]
    else:
        cbs_fav = np.nan
    rows.append({'away': away, 'home': home, 'fav': fav, 'dog': dog,
                 'spread': abs(sp.get(fav, np.nan)), 'p_fav': p_fav,
                 'cbs_fav': cbs_fav, 'tossup': abs(sp.get(fav, 0)) < TOSSUP,
                 'edge': (1 - p_fav) * cbs_fav / 100})
edge = pd.DataFrame(rows)

# ---- Entry A: pure chalk ----
edge['A'] = edge['fav']

# ---- Entry B: chalk, toss-ups go against the public, optional hunter flip ----
edge['B'] = edge['fav']
tu = edge['tossup'] & edge['cbs_fav'].notna()
edge.loc[tu & (edge['cbs_fav'] > 50), 'B'] = edge.loc[tu, 'dog']
edge['B_note'] = ''
edge.loc[tu & (edge['B'] != edge['A']), 'B_note'] = 'toss-up, fade public'
edge.loc[edge['tossup'] & edge['cbs_fav'].isna(), 'B_note'] = 'toss-up, need CBS %'
if B_MODE == 'hunter':
    cand = edge[(edge['B'] == edge['A']) & edge['edge'].notna()]
    if len(cand):
        i = cand['edge'].idxmax()
        edge.loc[i, 'B'] = edge.loc[i, 'dog']
        edge.loc[i, 'B_note'] = f"hunter flip (edge {edge.loc[i, 'edge']:.2f})"

print(f"Entry B mode: {B_MODE}.  B differs from A in {int((edge['A'] != edge['B']).sum())} game(s).\n")
for _, r in edge.iterrows():
    mark = '  <-- ' + r['B_note'] if r['B_note'] else ''
    print(f"{r['away']:>12} @ {r['home']:<12}  A: {r['A']:<12} B: {r['B']:<12}{mark}")

print("\nFlip candidates, ranked by edge = P(dog) x CBS% on favorite:")
(edge[['away', 'home', 'fav', 'spread', 'p_fav', 'cbs_fav', 'edge', 'tossup']]
     .sort_values('edge', ascending=False, na_position='last')
     .round(3))

Entry B mode: fuse.  B differs from A in 1 game(s).

    Patriots @ Seahawks      A: Seahawks     B: Seahawks    
       49ers @ Rams          A: Rams         B: Rams        
      Browns @ Jaguars       A: Jaguars      B: Jaguars     
  Buccaneers @ Bengals       A: Bengals      B: Bengals     
      Ravens @ Colts         A: Ravens       B: Ravens      
     Falcons @ Steelers      A: Steelers     B: Steelers    
       Bills @ Texans        A: Bills        B: Texans        <-- toss-up, fade public
       Bears @ Panthers      A: Bears        B: Bears       
        Jets @ Titans        A: Titans       B: Titans      
      Saints @ Lions         A: Lions        B: Lions       
   Cardinals @ Chargers      A: Chargers     B: Chargers    
    Dolphins @ Raiders       A: Raiders      B: Raiders     
  Commanders @ Eagles        A: Eagles       B: Eagles      
     Packers @ Vikings       A: Vikings      B: Vikings     
     Cowboys @ Giants        A: Cowboys      B: Cowboys     
     B

,away,home,fav,spread,p_fav,cbs_fav,edge,tossup
6,Bills,Texans,Bills,1.167,0.510,81,0.397,True
7,Bears,Panthers,Bears,2.556,0.580,87,0.365,False
8,Jets,Titans,Titans,1.944,0.551,80,0.359,False
5,Falcons,Steelers,Steelers,3.250,0.612,86,0.334,False
14,Cowboys,Giants,Cowboys,2.556,0.580,77,0.323,False
4,Ravens,Colts,Ravens,3.500,0.619,83,0.316,False
12,Commanders,Eagles,Eagles,5.188,0.665,93,0.311,False
1,49ers,Rams,Rams,3.611,0.639,86,0.311,False
11,Dolphins,Raiders,Raiders,3.500,0.628,81,0.301,False
3,Buccaneers,Bengals,Bengals,3.562,0.641,76,0.273,False


## Tiebreaker

Over/under total for the last matchup of the week, averaged across books.

In [9]:
tot = raw[raw['section'].isin(total_sections)].reset_index(drop=True).copy()
for s in sources:
    tot[s] = tot[s].map(parse_total)
tot['ave_total'] = tot[sources].mean(axis=1)
tot = tot.dropna(subset=['ave_total'])

# last game listed = last matchup of the week; its over/under rows are a pair
away, home = tot['Team'].iloc[-2], tot['Team'].iloc[-1]
tiebreak = tot['ave_total'].iloc[-2:].mean()
print(f"Tiebreaker: {away} @ {home}, vegas total = {tiebreak:.1f}")
# Pool rule is plain "closest". If A and B have identical picks they can both be in the
# same tiebreak, so straddle the total (~3 each side) to cover the distribution. If the
# picks differ they can't both tie, so each entry just wants the median: bracket the
# vegas total with the integers on either side (also avoids sitting exactly on the
# number vegas-following rivals will write, which would only split the tiebreak).
STRADDLE = 3
same_picks = (edge['A'] == edge['B']).all()
if same_picks:
    tb_A, tb_B = round(tiebreak - STRADDLE), round(tiebreak + STRADDLE)
else:
    tb_A, tb_B = int(np.floor(tiebreak)), int(np.floor(tiebreak)) + 1
print(f"  Entry A: {tb_A}   Entry B: {tb_B}   ({'identical picks, straddling' if same_picks else 'picks differ, bracketing'})")
tot[['Team'] + sources + ['ave_total']].iloc[-2:]

Tiebreaker: Broncos @ Chiefs, vegas total = 43.1
  Entry A: 40   Entry B: 46


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_total
30,Broncos,43.5,43.5,42.5,43.0,43.5,42.5,43.5,43.0,42.5,43.055556
31,Chiefs,43.5,43.5,42.5,43.0,43.5,42.5,43.5,43.0,42.5,43.055556


## Hannah's picks (Entry A)

One team per line, in site order, tiebreaker last. Copy/paste.

In [14]:
print("Hannah's picks:")
print("\n".join(edge['A']))
print(f"tiebreak {tb_A}")

Hannah's picks:
Seahawks
Rams
Jaguars
Bengals
Ravens
Steelers
Bills
Bears
Titans
Lions
Chargers
Raiders
Eagles
Vikings
Cowboys
Chiefs
tiebreak 40


## Sean's picks (Entry B)

In [15]:
print("Sean's picks:")
print("\n".join(edge['B']))
print(f"tiebreak {tb_B}")

Sean's picks:
Seahawks
Rams
Jaguars
Bengals
Ravens
Steelers
Texans
Bears
Titans
Lions
Chargers
Raiders
Eagles
Vikings
Cowboys
Chiefs
tiebreak 46
